<a href="https://colab.research.google.com/github/GauriNehe/Flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/GauriNehe/Flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

One row = one content_hash_id, observed on one report_date, within a monthly partition (month). I'm using month=2026-03 as my mid-panel window.

In [ ]:
!pip install duckdb --quiet
import duckdb, os
from google.colab import userdata

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

con = duckdb.connect()
con.sql("INSTALL httpfs;")
con.sql("LOAD httpfs;")

con.sql(f"""
CREATE SECRET hf_token (
    TYPE HUGGINGFACE,
    TOKEN '{os.environ["HF_TOKEN"]}'
);
""")

files = con.sql("SELECT * FROM glob('hf://datasets/FlyRank/internship-warehouse/**')").df()
print(files)
check1 = con.sql("""
    SELECT COUNT(*) AS total_rows, COUNT(DISTINCT content_hash_id) AS unique_content_ids
    FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet'
""").df()
print(check1)

                                                                                                      file
0                                                hf://datasets/FlyRank/internship-warehouse/.gitattributes
1                                                     hf://datasets/FlyRank/internship-warehouse/README.md
2                                           hf://datasets/FlyRank/internship-warehouse/dim_clients.parquet
3                                           hf://datasets/FlyRank/internship-warehouse/dim_content.parquet
4   hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2025-01/data_0.parquet
5   hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2025-02/data_0.parquet
6   hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2025-03/data_0.parquet
7   hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2025-04/data_0.parquet
8   hf://datasets/FlyRank/internship-

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   total_rows  unique_content_ids
0     9841378              331437


In [ ]:
import pandas as pd
pd.set_option('display.max_colwidth', None)
print(files)

                                                                                                      file
0                                                hf://datasets/FlyRank/internship-warehouse/.gitattributes
1                                                     hf://datasets/FlyRank/internship-warehouse/README.md
2                                           hf://datasets/FlyRank/internship-warehouse/dim_clients.parquet
3                                           hf://datasets/FlyRank/internship-warehouse/dim_content.parquet
4   hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2025-01/data_0.parquet
5   hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2025-02/data_0.parquet
6   hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2025-03/data_0.parquet
7   hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2025-04/data_0.parquet
8   hf://datasets/FlyRank/internship-

In [ ]:
con.sql("DESCRIBE SELECT * FROM 'hf://datasets/FlyRank/internship-warehouse/dim_content.parquet'").show()
print("---")
con.sql("DESCRIBE SELECT * FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet'").show()

┌────────────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│        column_name         │ column_type │  null   │   key   │ default │  extra  │
│          varchar           │   varchar   │ varchar │ varchar │ varchar │ varchar │
├────────────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ client_hash_id             │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ content_hash_id            │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ keyword_hash_id            │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ url_hash_id                │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ keyword_char_count         │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ keyword_token_count        │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ url_char_count             │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ content_created_date       │ DATE        │ YES     │ NULL    │ 

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

Features: gsc_impressions, gsc_clicks, gsc_sum_position, word_count, char_count, is_published (from dim_content), content age (derived from content_created_date)
Label/proxy: month-over-month change in gsc_impressions per content_hash_id
Context (kept for joins/grouping, not modeled): content_hash_id, client_hash_id, report_date
Excluded: sessions_ai and all ai_* columns (chatgpt, claude, gemini, copilot, perplexity, meta, other) — these are sparse/AI-referral traffic signals, risky to use as features since most rows will be null or zero. Also excluding scroll_events, since it depends on ga4_data_available which isn't true for every client.

In [ ]:
q2 = con.sql("""
    SELECT
        AVG(CASE WHEN sessions_ai IS NULL THEN 1 ELSE 0 END) AS pct_null_sessions_ai,
        AVG(CASE WHEN ga4_data_available THEN 1 ELSE 0 END) AS pct_ga4_available
    FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet'
""").df()
print(q2)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   pct_null_sessions_ai  pct_ga4_available
0               0.30674           0.042064


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

Three checks on month=2026-03: (1) grain — confirms one row per content per day, not per content total; (2) row count and date span — confirms the slice size and that dates fall within March 2026; (3) availability — filters to rows where GSC data actually exists, using IS TRUE, and shows how many survive.

In [ ]:
q3a = con.sql("""
    SELECT content_hash_id, COUNT(*) AS rows_per_content
    FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet'
    GROUP BY content_hash_id
    ORDER BY rows_per_content DESC
    LIMIT 5
""").df()
print(q3a)

            content_hash_id  rows_per_content
0  content_d0dff76c889de68f                31
1  content_67741cce996cfafa                31
2  content_2e6360ad20fd7107                31
3  content_ac8663da7484669a                31
4  content_65c50dfe9d87a585                31


In [ ]:
q3b = con.sql("""
    SELECT COUNT(*) AS row_count, MIN(report_date) AS earliest, MAX(report_date) AS latest
    FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet'
""").df()
print(q3b)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   row_count   earliest     latest
0    9841378 2026-03-01 2026-03-31


In [ ]:
q3c = con.sql("""
    SELECT COUNT(*) AS available_rows
    FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet'
    WHERE gsc_data_available IS TRUE
""").df()
print(q3c)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   available_rows
0         3611061


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

This data can't tell me why a page's performance changed — only that it did. Specifically: (1) Unbalanced history — only 36.7% of March 2026 rows have GSC data available, so any model trained on GSC signals will silently exclude nearly two-thirds of the slice; (2) GSC-only early rows — earlier months in the panel may lean more heavily on Search Console alone before GA4/AI-referral tracking matured, so trends across months aren't perfectly comparable; (3) Window overlaps — content_created_date and optimization_eligible_date can predate the panel's start, so "content age" features may undercount true age for pages that existed before this warehouse's history began.

In [ ]:
q4 = con.sql("""
    SELECT
        MIN(content_created_date) AS earliest_content_created
    FROM 'hf://datasets/FlyRank/internship-warehouse/dim_content.parquet'
""").df()
print(q4)

  earliest_content_created
0               2024-10-16


## Self-check

Before you submit, confirm each line honestly:

- [X] Every section above is filled — markdown thinking AND the code that backs it
- [X] The notebook runs top to bottom with no errors (Runtime → Run all)
- [X] No client names, URLs, or private queries anywhere
- [X] My claims use careful words: observed, measured, directional, decision-support
- [X] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.